# SARIMA Generator

Generate Seasonal ARIMA time series with configurable AR, MA, differencing, and seasonal components.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from synforecast.generators import SARIMAGenerator

## Example 1: Basic SARIMA(2,1,1)(1,1,1)_7

A SARIMA model with weekly seasonality, suitable for daily data.

In [ ]:
params = {
    "min_length": 200,
    "max_length": 200,
    "freq": "D",
    "p": 2,
    "d": 1,
    "q": 1,
    "P": 1,
    "D": 1,
    "Q": 1,
    "seasonal_period": 7,
    "noise_std": 2.0,
    "drift": 0.5,
    "seed": 42,
}

generator = SARIMAGenerator(engine="polars", **params)

### Model Information

Inspect the auto-generated model parameters and polynomial structure.

In [ ]:
model_info = generator.get_model_info()
print(f"Model: {model_info['model']}")
print(f"AR parameters: {model_info['ar_params']}")
print(f"MA parameters: {model_info['ma_params']}")
print(f"Seasonal AR parameters: {model_info['seasonal_ar_params']}")
print(f"Seasonal MA parameters: {model_info['seasonal_ma_params']}")
print(f"Drift: {model_info['drift']}")
print(f"Burn-in: {model_info['burn_in']}")
print(f"\nExpanded AR polynomial lags: {model_info['full_ar_polynomial_lags']}")
print(f"Expanded AR polynomial coeffs: {model_info['full_ar_polynomial_coeffs']}")

### Generate and Inspect Data

In [ ]:
df = generator.generate(n_series=3)
print(f"Generated {df['unique_id'].n_unique()} time series")
print(f"Total observations: {len(df)}")
df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df["unique_id"].unique().to_list():
    series = df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("SARIMA(2,1,1)(1,1,1)_7 Time Series")
ax.legend()
plt.tight_layout()
plt.show()

### Statistics by Series

In [ ]:
df.group_by("unique_id").agg(
    [
        pl.col("y").count().alias("count"),
        pl.col("y").min().alias("min_value"),
        pl.col("y").max().alias("max_value"),
        pl.col("y").mean().alias("mean_value"),
        pl.col("y").std().alias("std_value"),
    ]
).sort("unique_id")

## Example 2: Stationary ARMA(1,1) with Custom Parameters

A stationary model with no differencing and explicit AR/MA coefficients.

In [ ]:
params_arma = {
    "min_length": 200,
    "max_length": 200,
    "freq": "D",
    "p": 1,
    "d": 0,
    "q": 1,
    "P": 0,
    "D": 0,
    "Q": 0,
    "ar_params": [0.7],
    "ma_params": [0.3],
    "mean": 50.0,
    "noise_std": 1.0,
    "seed": 123,
}

generator_arma = SARIMAGenerator(engine="polars", **params_arma)
df_arma = generator_arma.generate(n_series=1)

print(f"Model: {generator_arma.get_model_info()['model']}")
print(f"Mean: {generator_arma.get_model_info()['mean']}")
print(f"Series mean: {df_arma['y'].mean():.2f}")
print(f"Series std: {df_arma['y'].std():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_arma["unique_id"].unique().to_list():
    series = df_arma.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("Stationary ARMA(1,1) Time Series")
ax.legend()
plt.tight_layout()
plt.show()

## Example 3: Pure Seasonal ARIMA(0,0,0)(1,1,1)_12

A purely seasonal model with monthly frequency and 12-month seasonality.

In [ ]:
params_seasonal = {
    "min_length": 100,
    "max_length": 100,
    "freq": "MS",
    "p": 0,
    "d": 0,
    "q": 0,
    "P": 1,
    "D": 1,
    "Q": 1,
    "seasonal_period": 12,
    "seasonal_ar_params": [0.5],
    "seasonal_ma_params": [0.3],
    "noise_std": 1.0,
    "seed": 456,
}

generator_seasonal = SARIMAGenerator(engine="polars", **params_seasonal)
df_seasonal = generator_seasonal.generate(n_series=1)

print(f"Model: {generator_seasonal.get_model_info()['model']}")
print("\nFirst 24 months:")
df_seasonal.head(24)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df_seasonal["unique_id"].unique().to_list():
    series = df_seasonal.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("Pure Seasonal ARIMA(0,0,0)(1,1,1)_12 Time Series")
ax.legend()
plt.tight_layout()
plt.show()